# Diagnostyka silników Diesla — pipeline AI

Notebook rozbija model z `diagnose.py` na osobne kroki, żeby dało się je czytać i edytować.

**Split etykiet**

- `val.csv` — trening (silniki, które model **widzi**)
- `final_valid.csv` — holdout (silniki, których **nie** ma w `val`; tu liczymy uczciwy Raw_Score)
- `test.csv` — submit bez etykiet

**Metryka hackathonu**

`Raw_Score = 0.75 * Macro-F1(label) + 0.25 * Accuracy(severity dla uszkodzonych)`

**Pomysł modelu**

1. Na kompletnym `val` / `final_valid` wstrzykujemy ~5% luk pomiarowych (jak `test.csv`), potem uzupełniamy NaN interpolacją.
2. Od każdego cylindra odejmujemy zdrowy profil *tego samego silnika* → residual.
3. RandomForest na widmie + residualu + sygnaturach (dołki/szczyty).
4. Twarde reguły akustyczne poprawiają 4 znane usterki i `unknown`.
5. Severity bierzemy z wielkości sygnatury, nie z drugiego modelu.


## 1. Importy i stałe

Tu zmieniasz hiperparametry lasu, próg `ok` vs `unknown` i cięcia severity.


In [1]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

BASE_DIR = Path(".").resolve()
FREQ_COLS = [f"mV_{i}" for i in range(21)]  # 0–20 kHz, krok 1 kHz
LABELS = ["ok", "zakoksowany", "lejacy", "pompa", "iglica", "unknown"]
FAULTS = ["zakoksowany", "lejacy", "pompa", "iglica"]
SEV_ORDER = ["male", "srednie", "duze"]
LABELED_COLS = ["engine_id", "cylinder", "n_cylinders", *FREQ_COLS, "label", "severity"]

# Residual L2 powyżej tego, przy głosie RF="ok", traktujemy jako unknown
OK_L2_MAX = 32.0

# val/final_valid są kompletne; test.csv ma ~5% luk. Wstrzykujemy ten sam wzorzec.
GAP_RATE = 0.05
GAP_SEED = 0

# Fallback, gdy w foldzie brakuje któregoś stopnia nasilenia
SEV_DEFAULTS = {
    "zakoksowany": (24.5, 35.5),  # peak12 (odbicie 9→12 kHz)
    "lejacy": (80.0, 118.0),      # L2 residualu
    "pompa": (39.0, 68.0),
    "iglica": (36.0, 60.0),
}

RF_PARAMS = dict(
    n_estimators=500,
    random_state=42,
    class_weight="balanced",
    min_samples_leaf=2,
    n_jobs=1,
)


def read_labeled_csv(path: Path) -> pd.DataFrame:
    """val / final_valid — działa też bez nagłówka."""
    header = pd.read_csv(path, nrows=0)
    if "engine_id" in header.columns:
        return pd.read_csv(path)
    return pd.read_csv(path, header=None, names=LABELED_COLS)


print("katalog:", BASE_DIR)
print("pasma częstotliwości:", len(FREQ_COLS), "kolumn,", FREQ_COLS[0], "…", FREQ_COLS[-1])
print("klasy:", LABELS)
print("usterki (severity male/srednie/duze):", FAULTS)


katalog: /home/janek/Desktop/hackathon-engin
pasma częstotliwości: 21 kolumn, mV_0 … mV_20
klasy: ['ok', 'zakoksowany', 'lejacy', 'pompa', 'iglica', 'unknown']
usterki (severity male/srednie/duze): ['zakoksowany', 'lejacy', 'pompa', 'iglica']


## 2. Wczytanie danych

- `val.csv` — trening z etykietami (`label` + `severity`)
- `final_valid.csv` — holdout z etykietami; **te silniki nie wchodzą do fitu** — tu weryfikujemy Raw_Score
- `train.csv` — 240 silników bez etykiet (szum + NaN); na razie nie trenujemy na nim
- `test.csv` — 50 silników bez etykiet, to idzie do `predictions.csv`


In [2]:
val = read_labeled_csv(BASE_DIR / "val.csv")
final_valid = read_labeled_csv(BASE_DIR / "final_valid.csv")
train = pd.read_csv(BASE_DIR / "train.csv")
test = pd.read_csv(BASE_DIR / "test.csv")

overlap = set(val["engine_id"]) & set(final_valid["engine_id"])
assert not overlap, f"wyciek silników val ∩ final_valid: {sorted(overlap)}"


def describe_set(name, df):
    n_eng = df["engine_id"].nunique()
    n_cyl = len(df)
    nan_pct = 100 * df[FREQ_COLS].isna().sum().sum() / (n_cyl * len(FREQ_COLS))
    sizes = df.groupby("engine_id")["n_cylinders"].first().value_counts().sort_index().to_dict()
    print(f"\n=== {name} ===")
    print(f"  wiersze (cylindry): {n_cyl}")
    print(f"  silniki:            {n_eng}")
    print(f"  rozmiary jednostek: {sizes}")
    print(f"  NaN w widmie:       {nan_pct:.2f}%")
    if "label" in df.columns:
        print("  etykiety:")
        print(df["label"].value_counts().reindex(LABELS).fillna(0).astype(int).to_string())
        print("  severity × label:")
        print(pd.crosstab(df["label"], df["severity"]).to_string())

describe_set("val (trening)", val)
describe_set("final_valid (holdout)", final_valid)
describe_set("train (bez etykiet)", train)
describe_set("test (submit)", test)
print("\nsilniki holdout:", sorted(final_valid["engine_id"].unique()))



=== val (trening) ===
  wiersze (cylindry): 328
  silniki:            28
  rozmiary jednostek: {8: 10, 12: 10, 16: 8}
  NaN w widmie:       0.00%
  etykiety:
label
ok             279
zakoksowany     12
lejacy          10
pompa            6
iglica          13
unknown          8
  severity × label:
severity     duze  male  nie_dotyczy  srednie
label                                        
iglica          2     6            0        5
lejacy          5     1            0        4
ok              0     0          279        0
pompa           0     4            0        2
unknown         0     0            8        0
zakoksowany     3     4            0        5

=== final_valid (holdout) ===
  wiersze (cylindry): 148
  silniki:            12
  rozmiary jednostek: {8: 4, 12: 3, 16: 5}
  NaN w widmie:       0.00%
  etykiety:
label
ok             128
zakoksowany      6
lejacy           4
pompa            3
iglica           3
unknown          4
  severity × label:
severity     duze  male  nie

## 3. Funkcje: czyszczenie, residual, cechy, reguły

Komórka z narzędziami. Pipeline niżej wywołuje je krok po kroku — tu edytujesz logikę sygnatur.


In [3]:
def _row_gap_seed(seed: int, engine_id: str, cylinder: int) -> int:
    payload = f"{int(seed)}:{engine_id}:{int(cylinder)}".encode()
    return int.from_bytes(hashlib.blake2b(payload, digest_size=8).digest(), "little")


def punch_spectrum_gaps(
    df: pd.DataFrame,
    *,
    rate: float = GAP_RATE,
    seed: int = GAP_SEED,
    verbose: bool = False,
) -> pd.DataFrame:
    """Zrzuca ~rate binów widma (jak train/test). Zbiory z NaN zostawia w spokoju."""
    out = df.copy()
    spec = out[FREQ_COLS].to_numpy(dtype=float, copy=True)
    if np.isnan(spec).any():
        if verbose:
            n_nan = int(np.isnan(spec).sum())
            print(
                f"  luki pomiarowe: już {n_nan}/{spec.size} "
                f"({100 * n_nan / spec.size:.2f}%) — nie dubluję"
            )
        return out
    if rate <= 0:
        return out
    n_rows, n_bins = spec.shape
    eids = out["engine_id"].astype(str).to_numpy()
    cyls = np.asarray(out["cylinder"], dtype=np.int64)
    drop = np.zeros((n_rows, n_bins), dtype=bool)
    for i in range(n_rows):
        rng = np.random.default_rng(_row_gap_seed(seed, eids[i], int(cyls[i])))
        mask = rng.random(n_bins) < rate
        if int((~mask).sum()) < 2:
            mask[:] = False
            n_drop = min(n_bins - 2, max(1, int(round(rate * n_bins))))
            mask[rng.choice(n_bins, size=n_drop, replace=False)] = True
        drop[i] = mask
    spec[drop] = np.nan
    out[FREQ_COLS] = spec
    if verbose:
        n_nan = int(drop.sum())
        n_hit = int(drop.any(axis=1).sum())
        print(
            f"  luki pomiarowe: {n_nan}/{spec.size} ({100 * n_nan / spec.size:.2f}%)  "
            f"wiersze z luką: {n_hit}/{n_rows}"
        )
    return out


def interpolate_spectrum(df: pd.DataFrame) -> pd.DataFrame:
    """NaN wzdłuż częstotliwości: interpolacja + ekstrapolacja na brzegach."""
    out = df.copy()
    out[FREQ_COLS] = (
        out[FREQ_COLS]
        .interpolate(axis=1, limit_direction="both")
        .clip(lower=0.0)
    )
    return out


def engine_baseline(spectra: np.ndarray, keep_frac: float = 0.7) -> np.ndarray:
    """Zdrowy profil silnika: mediana cylindrów najbliższych medianie całego silnika."""
    med = np.median(spectra, axis=0)
    dist = np.linalg.norm(spectra - med, axis=1)
    k = max(2, int(np.ceil(len(spectra) * keep_frac)))
    keep = np.argsort(dist)[:k]
    return np.median(spectra[keep], axis=0)


def compute_residuals(spectra: np.ndarray, engine_ids: np.ndarray) -> np.ndarray:
    residual = np.zeros_like(spectra, dtype=float)
    for eid in np.unique(engine_ids):
        idx = np.where(engine_ids == eid)[0]
        residual[idx] = spectra[idx] - engine_baseline(spectra[idx])
    return residual


def signature_table(residual: np.ndarray) -> dict[str, np.ndarray]:
    """Ręczne cechy z residualu — dołki/szczyty z wykresów usterkowych."""
    return {
        "l2": np.linalg.norm(residual, axis=1),
        "l1": np.abs(residual).sum(axis=1),
        "dip9": residual[:, 9] - 0.5 * (residual[:, 8] + residual[:, 10]),
        "peak12": residual[:, 12] - residual[:, 9],
        "dip3": residual[:, 3] - 0.5 * (residual[:, 2] + residual[:, 4]),
        "dip18": residual[:, 18] - 0.5 * (residual[:, 17] + residual[:, 19]),
        "hf": residual[:, 14:].mean(axis=1),
        "mf": residual[:, 6:12].mean(axis=1),
        "lf": residual[:, :5].mean(axis=1),
        "rough": np.abs(np.diff(residual, axis=1)).sum(axis=1),
    }


def build_templates(residual: np.ndarray, labels: np.ndarray) -> dict[str, np.ndarray]:
    """Średni residual każdej usterki — prototyp do cosine similarity."""
    templates = {}
    for lab in FAULTS:
        mask = labels == lab
        templates[lab] = residual[mask].mean(axis=0) if mask.any() else np.zeros(residual.shape[1])
    return templates


def cosine_to_templates(residual: np.ndarray, templates: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
    norms = np.clip(np.linalg.norm(residual, axis=1, keepdims=True), 1e-9, None)
    unit = residual / norms
    out = {}
    for lab, vec in templates.items():
        tnorm = max(float(np.linalg.norm(vec)), 1e-9)
        out[lab] = unit @ (vec / tnorm)
    return out


def feature_matrix(spectra, residual, sig) -> np.ndarray:
    unit = residual / np.clip(np.linalg.norm(residual, axis=1, keepdims=True), 1e-6, None)
    extra = np.column_stack([
        sig["dip9"], sig["peak12"], sig["dip3"], sig["dip18"],
        sig["hf"], sig["mf"], sig["lf"],
        sig["l2"], sig["l1"],
        residual[:, 9], residual[:, 3], residual[:, 12], residual[:, 18], residual[:, 19],
        sig["rough"], spectra.sum(axis=1),
    ])
    return np.hstack([spectra, residual, unit, extra])


def postprocess_labels(rf_pred, sig, cos) -> np.ndarray:
    """Nadpisuje RF, gdy sygnatura akustyczna jest jednoznaczna. Edytuj progi tutaj."""
    out = rf_pred.copy()
    for i in range(len(out)):
        lab = out[i]
        dip9, peak12, l2 = sig["dip9"][i], sig["peak12"][i], sig["l2"][i]
        hf, mf = sig["hf"][i], sig["mf"][i]

        if dip9 < -7.5 and peak12 > 16.0 and cos["zakoksowany"][i] > 0.80:
            lab = "zakoksowany"
        elif l2 > 60 and hf < -15 and mf < -18:
            lab = "lejacy"
        elif cos["pompa"][i] > 0.95 and l2 > 20 and dip9 > -5.0 and not (l2 > 60 and hf < -15):
            lab = "pompa"
        elif lab == "zakoksowany" and (dip9 > -6.0 or peak12 < 14.0 or cos["zakoksowany"][i] < 0.70):
            lab = "unknown"
        elif lab == "pompa" and cos["pompa"][i] < 0.55:
            lab = "unknown"
        elif lab == "ok" and cos["iglica"][i] > 0.95 and l2 > 22:
            lab = "iglica"
        elif lab == "ok" and l2 > OK_L2_MAX:
            lab = "unknown"
        elif lab == "lejacy" and l2 < 55:
            lab = "unknown"
        out[i] = lab
    return out


def severity_magnitude(sig):
    # koks: wielkość odbicia 9→12 kHz; reszta: energia residualu
    return {
        "zakoksowany": sig["peak12"],
        "lejacy": sig["l2"],
        "pompa": sig["l2"],
        "iglica": sig["l2"],
    }


def fit_severity_thresholds(labels, severity, mag):
    """Cięcia w połowie luki między male/srednie i srednie/duze."""
    thr = {}
    for lab in FAULTS:
        groups = {sev: mag[lab][(labels == lab) & (severity == sev)] for sev in SEV_ORDER}
        t1_def, t2_def = SEV_DEFAULTS[lab]
        if len(groups["male"]) and len(groups["srednie"]):
            t1 = 0.5 * (float(groups["male"].max()) + float(groups["srednie"].min()))
        else:
            t1 = t1_def
        if len(groups["srednie"]) and len(groups["duze"]):
            t2 = 0.5 * (float(groups["srednie"].max()) + float(groups["duze"].min()))
        else:
            t2 = t2_def
        if t2 <= t1:
            t1, t2 = t1_def, t2_def
        thr[lab] = (t1, t2)
    return thr


def apply_severity(pred_label, mag, thresholds):
    out = np.full(len(pred_label), "nie_dotyczy", dtype=object)
    for i, lab in enumerate(pred_label):
        if lab not in FAULTS:
            continue
        t1, t2 = thresholds[lab]
        v = mag[lab][i]
        if v < t1:
            out[i] = "male"
        elif v < t2:
            out[i] = "srednie"
        else:
            out[i] = "duze"
    return out


def hackathon_score(y_true, y_pred, s_true, s_pred):
    macro = f1_score(y_true, y_pred, average="macro", labels=LABELS)
    mask = np.isin(y_true, FAULTS)
    sev_acc = accuracy_score(s_true[mask], s_pred[mask]) if mask.any() else 1.0
    raw = 0.75 * macro + 0.25 * sev_acc
    return raw, float(macro), float(sev_acc)


def prepare_xy(df, templates=None):
    """Z DataFrame → widmo, residual, sygnatury, cechy RF, cosine (jeśli są szablony)."""
    df = interpolate_spectrum(df)
    spectra = df[FREQ_COLS].to_numpy(float)
    engine_ids = df["engine_id"].to_numpy()
    residual = compute_residuals(spectra, engine_ids)
    sig = signature_table(residual)
    feats = feature_matrix(spectra, residual, sig)
    cos = cosine_to_templates(residual, templates) if templates is not None else None
    return df, spectra, residual, sig, feats, cos

print("funkcje załadowane")


funkcje załadowane


## 4. Preprocessing `val` — luki warsztatowe, interpolacja i residuale

`val.csv` jest pomiarem laboratoryjnym (0% NaN). `test.csv` ma ~5% luk, więc zanim liczymy residuale, wstrzykujemy ten sam wzorzec dziur i dopiero wtedy interpolujemy. Cylindry **tego samego silnika** są porównywane ze sobą.


In [4]:
print("symulacja luk warsztatowych na val (jak test.csv, ~5%):")
val = punch_spectrum_gaps(val, verbose=True)
val_clean = interpolate_spectrum(val)
nan_before = val[FREQ_COLS].isna().sum().sum()
nan_after = val_clean[FREQ_COLS].isna().sum().sum()
print(f"NaN w val: {nan_before} → po interpolacji {nan_after}")

spectra_val = val_clean[FREQ_COLS].to_numpy(float)
residual_val = compute_residuals(spectra_val, val_clean["engine_id"].to_numpy())
sig_val = signature_table(residual_val)

overview = val_clean[["engine_id", "cylinder", "label", "severity"]].copy()
overview["l2"] = sig_val["l2"]
overview["dip9"] = sig_val["dip9"]
overview["peak12"] = sig_val["peak12"]
overview["dip3"] = sig_val["dip3"]
overview["hf"] = sig_val["hf"]
overview["mf"] = sig_val["mf"]

print("\nL2 residualu vs mediana silnika (im wyżej, tym bardziej cylinder odstaje):")
print(
    overview.groupby(["label", "severity"])["l2"]
    .agg(n="count", mean="mean", min="min", max="max")
    .round(2)
    .to_string()
)

print("\nSygnatury (średnia residualu) — zakoksowany = dołek 9 kHz + szczyt 12 kHz:")
print(
    overview.groupby("label")[["dip9", "peak12", "dip3", "hf", "mf", "l2"]]
    .mean()
    .round(2)
    .reindex(LABELS)
    .to_string()
)


symulacja luk warsztatowych na val (jak test.csv, ~5%):
  luki pomiarowe: 311/6888 (4.52%)  wiersze z luką: 201/328
NaN w val: 311 → po interpolacji 0

L2 residualu vs mediana silnika (im wyżej, tym bardziej cylinder odstaje):
                           n    mean     min     max
label       severity                                
iglica      duze           2   69.33   68.95   69.72
            male           6   28.94   24.93   35.88
            srednie        5   44.49   35.96   51.51
lejacy      duze           5  140.41  134.74  148.78
            male           1   68.12   68.12   68.12
            srednie        4   95.88   89.83  100.90
ok          nie_dotyczy  279   11.64    2.78   31.55
pompa       male           4   27.08   23.25   33.49
            srednie        2   54.91   53.38   56.44
unknown     nie_dotyczy    8   84.84   53.69  126.64
zakoksowany duze           3   60.07   58.09   61.78
            male           4   32.27   28.08   36.26
            srednie        5   

## 5. Trening = `val.csv`, test = `final_valid.csv`

Cylindry z jednego silnika są skorelowane, więc holdout jest po **całych silnikach**. `final_valid.csv` to te jednostki, których nie ma w `val.csv`. Holdout też dostaje luki (~5%), bo jury ocenia na dziurawym `test.csv`.


In [5]:
y_all = val_clean["label"].to_numpy()
s_all = val_clean["severity"].to_numpy()

val_tr = val_clean.reset_index(drop=True)
print("symulacja luk warsztatowych na final_valid:")
val_ho_raw = punch_spectrum_gaps(final_valid, verbose=True)
val_ho = interpolate_spectrum(val_ho_raw).reset_index(drop=True)

print("train = val.csv   |   holdout = final_valid.csv")
print(f"  train:   {val_tr['engine_id'].nunique():2d} silników, {len(val_tr):3d} cylindrów")
print(f"  holdout: {val_ho['engine_id'].nunique():2d} silników, {len(val_ho):3d} cylindrów")
print("  silniki holdout:", sorted(val_ho["engine_id"].unique()))
print("\nrozkład label w train:")
print(val_tr["label"].value_counts().reindex(LABELS).fillna(0).astype(int).to_string())
print("\nrozkład label w holdout:")
print(val_ho["label"].value_counts().reindex(LABELS).fillna(0).astype(int).to_string())


symulacja luk warsztatowych na final_valid:
  luki pomiarowe: 172/3108 (5.53%)  wiersze z luką: 100/148
train = val.csv   |   holdout = final_valid.csv
  train:   28 silników, 328 cylindrów
  holdout: 12 silników, 148 cylindrów
  silniki holdout: ['val_0000', 'val_0001', 'val_0002', 'val_0003', 'val_0004', 'val_0005', 'val_0006', 'val_0007', 'val_0008', 'val_0009', 'val_0010', 'val_0011']

rozkład label w train:
label
ok             279
zakoksowany     12
lejacy          10
pompa            6
iglica          13
unknown          8

rozkład label w holdout:
label
ok             128
zakoksowany      6
lejacy           4
pompa            3
iglica           3
unknown          4


## 6. Cechy + szablony usterkowe (tylko z części treningowej)


In [6]:
_, spec_tr, res_tr, sig_tr, X_tr, _ = prepare_xy(val_tr)
templates = build_templates(res_tr, val_tr["label"].to_numpy())
cos_tr = cosine_to_templates(res_tr, templates)

y_tr = val_tr["label"].to_numpy()
s_tr = val_tr["severity"].to_numpy()

print(f"macierz cech train: {X_tr.shape[0]} próbek × {X_tr.shape[1]} cech")
print("  = 21 widmo + 21 residual + 21 kierunek residualu + 16 sygnatur")
print("\nszablony (średni residual) — norma L2:")
for lab, vec in templates.items():
    print(f"  {lab:12s}  ||template|| = {np.linalg.norm(vec):6.2f}")

print("\nśrednie cosine train → własny szablon:")
for lab in FAULTS:
    mask = y_tr == lab
    if mask.any():
        print(f"  {lab:12s}  mean cos = {cos_tr[lab][mask].mean():.3f}")


macierz cech train: 328 próbek × 79 cech
  = 21 widmo + 21 residual + 21 kierunek residualu + 16 sygnatur

szablony (średni residual) — norma L2:
  zakoksowany   ||template|| =  41.46
  lejacy        ||template|| = 114.12
  pompa         ||template|| =  32.93
  iglica        ||template|| =  40.09

średnie cosine train → własny szablon:
  zakoksowany   mean cos = 0.928
  lejacy        mean cos = 0.989
  pompa         mean cos = 0.906
  iglica        mean cos = 0.971


## 7. Trening RandomForest


In [7]:
rf = RandomForestClassifier(**RF_PARAMS)
rf.fit(X_tr, y_tr)

print("RandomForest wytrenowany")
print("  parametry:", RF_PARAMS)
print("  klasy modelu:", list(rf.classes_))
print("  n_estimators:", rf.n_estimators)

# szybki podgląd, czego las używa (nie interpretuj zbyt dosłownie)
feat_names = (
    [f"raw_{i}" for i in range(21)]
    + [f"res_{i}" for i in range(21)]
    + [f"dir_{i}" for i in range(21)]
    + [
        "dip9", "peak12", "dip3", "dip18", "hf", "mf", "lf", "l2", "l1",
        "res9", "res3", "res12", "res18", "res19", "rough", "energy",
    ]
)
imp = pd.Series(rf.feature_importances_, index=feat_names).sort_values(ascending=False)
print("\ntop 12 cech (importance):")
print(imp.head(12).round(4).to_string())


RandomForest wytrenowany
  parametry: {'n_estimators': 500, 'random_state': 42, 'class_weight': 'balanced', 'min_samples_leaf': 2, 'n_jobs': 1}
  klasy modelu: ['iglica', 'lejacy', 'ok', 'pompa', 'unknown', 'zakoksowany']
  n_estimators: 500

top 12 cech (importance):
dir_13    0.0438
rough     0.0425
peak12    0.0423
res12     0.0374
hf        0.0366
l1        0.0332
res_13    0.0314
res_12    0.0298
l2        0.0296
dip9      0.0293
res_17    0.0267
res_16    0.0232


## 8. Predykcja na `final_valid.csv` + reguły akustyczne


In [8]:
_, spec_ho, res_ho, sig_ho, X_ho, cos_ho = prepare_xy(val_ho, templates=templates)
y_ho = val_ho["label"].to_numpy()
s_ho = val_ho["severity"].to_numpy()

rf_pred_ho = rf.predict(X_ho)
y_pred_ho = postprocess_labels(rf_pred_ho, sig_ho, cos_ho)

n_changed = int((rf_pred_ho != y_pred_ho).sum())
print(f"RF surowy vs po regułach: {n_changed} cylindrów nadpisanych")
if n_changed:
    chg = pd.DataFrame({
        "engine_id": val_ho["engine_id"].to_numpy()[rf_pred_ho != y_pred_ho],
        "cylinder": val_ho["cylinder"].to_numpy()[rf_pred_ho != y_pred_ho],
        "rf": rf_pred_ho[rf_pred_ho != y_pred_ho],
        "po_regulach": y_pred_ho[rf_pred_ho != y_pred_ho],
        "true": y_ho[rf_pred_ho != y_pred_ho],
    })
    print(chg.to_string(index=False))

print("\n=== classification report (holdout, po regułach) ===")
print(classification_report(y_ho, y_pred_ho, labels=LABELS, digits=3, zero_division=0))
print("macierz pomyłek  wiersz=true  kolumna=pred")
print(pd.DataFrame(
    confusion_matrix(y_ho, y_pred_ho, labels=LABELS),
    index=LABELS,
    columns=LABELS,
).to_string())


RF surowy vs po regułach: 1 cylindrów nadpisanych
engine_id  cylinder rf po_regulach    true
 val_0001        16 ok     unknown unknown

=== classification report (holdout, po regułach) ===
              precision    recall  f1-score   support

          ok      1.000     1.000     1.000       128
 zakoksowany      1.000     1.000     1.000         6
      lejacy      1.000     1.000     1.000         4
       pompa      1.000     1.000     1.000         3
      iglica      1.000     1.000     1.000         3
     unknown      1.000     1.000     1.000         4

    accuracy                          1.000       148
   macro avg      1.000     1.000     1.000       148
weighted avg      1.000     1.000     1.000       148

macierz pomyłek  wiersz=true  kolumna=pred
              ok  zakoksowany  lejacy  pompa  iglica  unknown
ok           128            0       0      0       0        0
zakoksowany    0            6       0      0       0        0
lejacy         0            0       4 

## 9. Severity na `final_valid.csv`

Cięcia uczymy **tylko na val**, potem przykładamy do holdoutu. To jest metryka, której używa jury (tylko na innym zbiorze).


In [9]:
mag_tr = severity_magnitude(sig_tr)
mag_ho = severity_magnitude(sig_ho)
sev_thr = fit_severity_thresholds(y_tr, s_tr, mag_tr)
s_pred_ho = apply_severity(y_pred_ho, mag_ho, sev_thr)

print("progi severity (t1: male→srednie,  t2: srednie→duze):")
for lab, (t1, t2) in sev_thr.items():
    feat = "peak12" if lab == "zakoksowany" else "l2"
    print(f"  {lab:12s}  {feat:7s}  |  {t1:6.2f}  |  {t2:6.2f}")

raw, macro, sev_acc = hackathon_score(y_ho, y_pred_ho, s_ho, s_pred_ho)
print(f"\n=== final_valid.csv  (held-out, nie było w ficie) ===")
print(f"  Macro-F1(label)                 = {macro:.4f}")
print(f"  Accuracy(severity | usterka)    = {sev_acc:.4f}")
print(f"  Raw_Score                       = {raw:.4f}")
print("  (jury liczy to samo na ukrytym test.csv)")

fault_mask = np.isin(y_ho, FAULTS)
if fault_mask.any():
    print("\nseverity na prawdziwych usterkach holdoutu:")
    print(pd.crosstab(
        pd.Series(s_ho[fault_mask], name="true"),
        pd.Series(s_pred_ho[fault_mask], name="pred"),
        dropna=False,
    ).to_string())


progi severity (t1: male→srednie,  t2: srednie→duze):
  zakoksowany   peak12   |   19.62  |   34.57
  lejacy        l2       |   78.97  |  117.82
  pompa         l2       |   43.43  |   68.00
  iglica        l2       |   35.92  |   60.23

=== final_valid.csv  (held-out, nie było w ficie) ===
  Macro-F1(label)                 = 1.0000
  Accuracy(severity | usterka)    = 0.9375
  Raw_Score                       = 0.9844
  (jury liczy to samo na ukrytym test.csv)

severity na prawdziwych usterkach holdoutu:
pred     duze  male  srednie
true                        
duze        2     0        0
male        0     4        1
srednie     0     0        9


## 10. Sanity na `val.csv` (resubstitution)

Las, szablony i progi z sekcji 6–9 są już na **całym** `val.csv`. Poniżej tylko górne ograniczenie (nie mylić z `final_valid`). Ten sam model idzie na `test.csv`.


In [10]:
# ten sam model co na final_valid — fit był na całym val.csv
templates_full = templates
rf_full = rf
sev_thr_full = sev_thr

_, spec_full, res_full, sig_full, X_full, _ = prepare_xy(val_clean)
y_full = val_clean["label"].to_numpy()
s_full = val_clean["severity"].to_numpy()
cos_full = cosine_to_templates(res_full, templates_full)
y_hat_full = postprocess_labels(rf_full.predict(X_full), sig_full, cos_full)
s_hat_full = apply_severity(y_hat_full, severity_magnitude(sig_full), sev_thr_full)
raw_f, mac_f, sev_f = hackathon_score(y_full, y_hat_full, s_full, s_hat_full)

print("model = fit na val.csv (użyty też na final_valid i test)")
print(f"  próbki: {X_full.shape[0]}  cechy: {X_full.shape[1]}")
print("  progi severity:")
for lab, (t1, t2) in sev_thr_full.items():
    print(f"    {lab:12s}  {t1:6.2f}  /  {t2:6.2f}")
print(f"\nresubstitution na val (górne ograniczenie, nie test):")
print(f"  Macro-F1={mac_f:.4f}  severity={sev_f:.4f}  Raw_Score={raw_f:.4f}")
print("uczciwy score jest w sekcji 9 (final_valid.csv)")


model = fit na val.csv (użyty też na final_valid i test)
  próbki: 328  cechy: 79
  progi severity:
    zakoksowany    19.62  /   34.57
    lejacy         78.97  /  117.82
    pompa          43.43  /   68.00
    iglica         35.92  /   60.23

resubstitution na val (górne ograniczenie, nie test):
  Macro-F1=0.9829  severity=0.9268  Raw_Score=0.9689
uczciwy score jest w sekcji 9 (final_valid.csv)


## 11. Predykcja `test.csv` i zapis `predictions.csv`

Ten sam model co na `final_valid` (trenowany wyłącznie na `val.csv`).


In [11]:
test_prep, spec_te, res_te, sig_te, X_te, cos_te = prepare_xy(test, templates=templates_full)
rf_pred_te = rf_full.predict(X_te)
y_te = postprocess_labels(rf_pred_te, sig_te, cos_te)
s_te = apply_severity(y_te, severity_magnitude(sig_te), sev_thr_full)

submit = pd.DataFrame({
    "engine_id": test_prep["engine_id"].to_numpy(),
    "cylinder": test_prep["cylinder"].to_numpy(),
    "label": y_te,
    "severity": s_te,
})

out_path = BASE_DIR / "predictions.csv"
submit.to_csv(out_path, index=False)

print(f"zapisano {out_path}  ({len(submit)} wierszy)")
print("\nrozkład label na teście:")
print(submit["label"].value_counts().reindex(LABELS).fillna(0).astype(int).to_string())
print("\nlabel × severity:")
print(pd.crosstab(submit["label"], submit["severity"]).to_string())

n_rf_changed = int((rf_pred_te != y_te).sum())
print(f"\nreguły nadpisały RF na teście: {n_rf_changed} cylindrów")

# format submitu
assert list(submit.columns) == ["engine_id", "cylinder", "label", "severity"]
assert len(submit) == len(test)
assert (submit["engine_id"].to_numpy() == test["engine_id"].to_numpy()).all()
assert (submit["cylinder"].to_numpy() == test["cylinder"].to_numpy()).all()
bad_ok = submit["label"].isin(["ok", "unknown"]) & (submit["severity"] != "nie_dotyczy")
bad_fault = submit["label"].isin(FAULTS) & ~submit["severity"].isin(SEV_ORDER)
assert not bad_ok.any() and not bad_fault.any(), "zła para label/severity"
print("\nformat sample_submit: OK")
submit.head(12)


zapisano /home/janek/Desktop/hackathon-engin/predictions.csv  (600 wierszy)

rozkład label na teście:
label
ok             518
zakoksowany     16
lejacy          13
pompa           13
iglica          18
unknown         22

label × severity:
severity     duze  male  nie_dotyczy  srednie
label                                        
iglica          5     3            0       10
lejacy          4     6            0        3
ok              0     0          518        0
pompa           2     5            0        6
unknown         0     0           22        0
zakoksowany     3     1            0       12

reguły nadpisały RF na teście: 5 cylindrów

format sample_submit: OK


,engine_id,cylinder,label,severity
0,test_0000,1,ok,nie_dotyczy
1,test_0000,2,ok,nie_dotyczy
2,test_0000,3,ok,nie_dotyczy
3,test_0000,4,ok,nie_dotyczy
4,test_0000,5,ok,nie_dotyczy
5,test_0000,6,ok,nie_dotyczy
6,test_0000,7,ok,nie_dotyczy
7,test_0000,8,ok,nie_dotyczy
8,test_0000,9,ok,nie_dotyczy
9,test_0000,10,ok,nie_dotyczy


## 12. (opcjonalnie) Leave-one-engine-out na `val ∪ final_valid`

Plików CSV **nie ruszamy**. W RAM sklejamy `val.csv` i `final_valid.csv`, wstrzykujemy luki jak w `test.csv`, potem LOEO po całym silniku: szablony, las i progi severity nie widzą held-out jednostki.

To jest uczciwsza ocena niż sam `final_valid` (40 silników, każda usterka kiedyś wypada). Ustaw `RUN_LOEO = True` i odpal komórkę.


In [12]:
RUN_LOEO = True  # True → concat val ∪ final_valid w RAM, LOEO po silniku

if not RUN_LOEO:
    print("pominięte — ustaw RUN_LOEO = True")
else:
    val_loeo = read_labeled_csv(BASE_DIR / "val.csv")
    ho_loeo = read_labeled_csv(BASE_DIR / "final_valid.csv")
    labeled = pd.concat([val_loeo, ho_loeo], ignore_index=True)
    labeled = punch_spectrum_gaps(labeled, verbose=True)
    labeled = interpolate_spectrum(labeled)
    spectra = labeled[FREQ_COLS].to_numpy(float)
    eids = labeled["engine_id"].to_numpy()
    residual = compute_residuals(spectra, eids)
    sig = signature_table(residual)
    y_lab = labeled["label"].to_numpy()
    s_lab = labeled["severity"].to_numpy()
    origin = np.array(["val"] * len(val_loeo) + ["final_valid"] * len(ho_loeo))
    engines_u = np.unique(eids)
    pred_y = np.empty(len(labeled), dtype=object)
    pred_s = np.empty(len(labeled), dtype=object)

    print("LOEO na val ∪ final_valid  (concat tylko w RAM, plików nie zapisujemy)")
    print(f"  {len(engines_u)} silników, {len(labeled)} cylindrów")
    print(f"  z val: {len(val_loeo)}  z final_valid: {len(ho_loeo)}")

    for i, eid in enumerate(engines_u, 1):
        tr = eids != eid
        te = eids == eid
        templates_i = build_templates(residual[tr], y_lab[tr])
        cos_te = cosine_to_templates(residual[te], templates_i)
        Xtr = feature_matrix(spectra[tr], residual[tr], {k: v[tr] for k, v in sig.items()})
        Xte = feature_matrix(spectra[te], residual[te], {k: v[te] for k, v in sig.items()})
        m = RandomForestClassifier(**RF_PARAMS)
        m.fit(Xtr, y_lab[tr])
        yhat = postprocess_labels(m.predict(Xte), {k: v[te] for k, v in sig.items()}, cos_te)
        mag = severity_magnitude(sig)
        thr = fit_severity_thresholds(y_lab[tr], s_lab[tr], {k: v[tr] for k, v in mag.items()})
        shat = apply_severity(yhat, {k: v[te] for k, v in mag.items()}, thr)
        pred_y[te] = yhat
        pred_s[te] = shat
        if i % 10 == 0 or i == len(engines_u):
            print(f"  {i}/{len(engines_u)}")

    raw, macro, sev_acc = hackathon_score(y_lab, pred_y, s_lab, pred_s)
    print("\n=== LOEO  RandomForest  (val ∪ final_valid) ===")
    print(classification_report(y_lab, pred_y, labels=LABELS, digits=3, zero_division=0))
    print(f"Macro-F1={macro:.4f}  severity={sev_acc:.4f}  Raw_Score={raw:.4f}")
    err = pred_y != y_lab
    if err.any():
        print(f"\nbłędy label ({int(err.sum())}):")
        print(
            pd.DataFrame({
                "split": origin[err],
                "engine_id": eids[err],
                "cylinder": labeled["cylinder"].to_numpy()[err],
                "true": y_lab[err],
                "pred": pred_y[err],
            }).to_string(index=False)
        )


  luki pomiarowe: 483/9996 (4.83%)  wiersze z luką: 301/476
LOEO na val ∪ final_valid  (concat tylko w RAM, plików nie zapisujemy)
  40 silników, 476 cylindrów
  z val: 328  z final_valid: 148
  10/40
  20/40
  30/40
  40/40

=== LOEO  RandomForest  (val ∪ final_valid) ===
              precision    recall  f1-score   support

          ok      0.998     1.000     0.999       407
 zakoksowany      1.000     0.944     0.971        18
      lejacy      0.933     1.000     0.966        14
       pompa      1.000     0.889     0.941         9
      iglica      1.000     0.938     0.968        16
     unknown      0.923     1.000     0.960        12

    accuracy                          0.994       476
   macro avg      0.976     0.962     0.967       476
weighted avg      0.994     0.994     0.994       476

Macro-F1=0.9674  severity=0.8596  Raw_Score=0.9405

błędy label (3):
split engine_id  cylinder        true    pred
  val  val_0012         7       pompa      ok
  val  val_0018       